# Task 1 — LLM-Powered Equity Research Assistant**CDAZZDEV Senior Machine Learning Engineer Assessment**This notebook runs the complete Task 1 pipeline end to end against live market data.| Section | Deliverable | Marks ||---|---|---|| 1A | OHLCV fetch, five indicators from first principles, news retrieval, summary dictionary | 60 || 1B | Per-headline sentiment JSON, LLM signal reasoning, Pydantic validation, prompt separation | 40 || Bonus | One-page rendered research brief with embedded chart | +5 |**Architecture.** All logic lives in `task1_financial/src/` as importable modules; thisnotebook is a thin execution harness. That is deliberate — logic buried in notebook cellscannot be unit-tested, and Section 2 of the brief says every part of the submission must bedefensible. Section 2 of this notebook runs the actual test suite so you can see theindicator maths verified against an independent implementation before any of it is used.**Reproducibility.** Every run prints a manifest: which provider and model served therequest, and which prompt versions were used.

## 0 · Environment setup`yfinance` is pinned to a minimum rather than an exact version because its news payloadschema changes between releases; `data_pipeline._news_from_yfinance` handles both the oldflat shape and the newer nested `content` shape.

In [ ]:
%%capture!pip install -q "yfinance>=0.2.40" "openai>=1.40" "pydantic>=2.7" jinja2 matplotlib pandas numpy requests

In [ ]:
import os, sys, json, logging, warningsfrom pathlib import Pathwarnings.filterwarnings("ignore", category=FutureWarning)logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s | %(message)s")logging.getLogger("matplotlib").setLevel(logging.WARNING)# --- Locate the repository -------------------------------------------------# On Colab: clone it. Locally (VS Code): the notebook already sits inside it.REPO = "CDAZZDEV-MLE-Udith"if not Path(REPO).exists() and not (Path.cwd().name == REPO or (Path.cwd() / "common").exists()):    !git clone -q https://github.com/YOUR_USERNAME/{REPO}.git    %cd {REPO}elif Path(REPO).exists() and Path.cwd().name != REPO:    %cd {REPO}ROOT = Path.cwd()sys.path[:0] = [str(ROOT), str(ROOT / "task1_financial" / "src")]print("Repository root:", ROOT)print("Contents:", sorted(p.name for p in ROOT.iterdir() if not p.name.startswith(".")))

### API keysKeys are read from Colab **Secrets** (the key icon in the left sidebar), never from a codecell. Add at least one of `GROQ_API_KEY` or `OPENROUTER_API_KEY` and toggle *Notebook access* on.Section 2.3 of the brief lists a hardcoded key as an automatic disqualification, so`common.llm_client.get_secret()` has no code path that accepts a literal key — it resolvesenvironment → Colab Secrets → interactive prompt, in that order.* Groq (recommended, free, no card): https://console.groq.com/keys* OpenRouter (fallback, free models): https://openrouter.ai/keys

In [ ]:
from common.llm_client import get_secretfor name in ("GROQ_API_KEY", "OPENROUTER_API_KEY"):    value = get_secret(name)    print(f"{name:<20} {'configured (' + value[:6] + '…)' if value else 'NOT SET'}")if not (os.environ.get("GROQ_API_KEY") or os.environ.get("OPENROUTER_API_KEY")):    raise SystemExit("Set at least one key in Colab Secrets, then re-run this cell.")

## 1 · ConfigurationThe ticker is the only thing you need to change. Nothing downstream is hardcoded to it,and there are **no literal date strings anywhere** — the window is expressed in years andresolved against `pd.Timestamp.now()` at call time.

In [ ]:
TICKER          = "NVDA"     # Try AAPL, MSFT, JPM, TSLA, or a non-US symbol like BARC.LLOOKBACK_YEARS  = 2          # Brief requires a minimum of two yearsMIN_HEADLINES   = 10         # Brief requires at least tenOUTPUT_DIR = ROOT / "task1_financial" / "output"OUTPUT_DIR.mkdir(parents=True, exist_ok=True)print(f"Analysing {TICKER} over {LOOKBACK_YEARS}y, targeting >= {MIN_HEADLINES} headlines.")

## 2 · Verifying the indicator maths *before* using itThe 25-mark criterion is "all five indicators computed correctly from first principles."Asserting that in prose is worth nothing, so the test suite verifies each indicator againstan **independent pure-Python reference implementation** — explicit loops over lists, nopandas, no numpy. Two independent derivations of the same textbook formula agreeing to1e-9 is real evidence; verifying pandas code with more pandas code sharing the sameassumptions is not.Three correctness decisions this catches, each of which the common implementation gets wrong:1. **RSI uses Wilder's smoothing, not a rolling mean.** `gain.rolling(14).mean()` is   *Cutler's* RSI — a different indicator. The test measures the divergence.2. **MACD uses `adjust=False` EMAs.** pandas defaults to `adjust=True`, which puts the   signal-line crossovers in the wrong places.3. **Bollinger Bands use the population σ (`ddof=0`).** pandas defaults to `ddof=1`, which   inflates a 2σ band by ~2.6%.

In [ ]:
!python task1_financial/tests/test_indicators.py

In [ ]:
!python task1_financial/tests/test_schemas.py

## 3 · Task 1A — data pipelineFetches split- and dividend-adjusted OHLCV (`auto_adjust=True`, a correctness requirement:an unadjusted series puts a −50% single-bar gap at a stock split, which detonates RSI anddrags the SMA-200 for 200 sessions), computes all five indicators, retrieves news througha four-source fallback chain, and assembles the summary dictionary.

In [ ]:
from data_pipeline import build_equity_datasetequity = build_equity_dataset(TICKER, lookback_years=LOOKBACK_YEARS, min_headlines=MIN_HEADLINES)print(f"\nSessions retrieved : {len(equity.prices)}")print(f"Date range         : {equity.prices.index[0].date()} → {equity.prices.index[-1].date()}")print(f"Headlines retrieved: {len(equity.news)}")if equity.warnings:    print("\nPipeline warnings (degradations, not failures):")    for w in equity.warnings:        print("  •", w)

### 3.1 Indicator outputThe tail of the enriched frame. Note the warm-up NaNs are never forward-filled — afabricated 200-day SMA on day 3 is a silent lie that would propagate into the signal.

In [ ]:
import pandas as pdpd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 3)cols = ["Close", "sma_50", "sma_200", "rsi_14", "macd", "macd_signal", "macd_hist",        "bb_lower", "bb_middle", "bb_upper", "bb_pct_b", "volatility_30d"]display(equity.prices[cols].tail(8))print("\nWarm-up (first valid observation per indicator):")for c in cols[1:]:    first = equity.prices[c].first_valid_index()    pos = equity.prices.index.get_loc(first) if first is not None else None    print(f"  {c:<16} row {pos:>4}  ({first.date() if first is not None else 'never'})")

### 3.2 News retrievalThe chain tries yfinance → Yahoo RSS → Google News RSS → NewsAPI, deduplicating bynormalised title and stopping as soon as the minimum is met. A single source is notenough: yfinance's payload shape changed between releases and any one feed can returnfewer than ten items on a quiet day.

In [ ]:
print(f"{len(equity.news)} unique headlines from "      f"{len({n.source for n in equity.news})} distinct source(s)\n")for i, item in enumerate(equity.news, 1):    print(f"{i:>2}. [{item.source[:22]:<22}] {item.headline[:96]}")

### 3.3 Summary dictionaryEvery required field, and every value is either a finite number or an explicit `None` —never `NaN`. That matters: handing a model the literal string `NaN` is a reliable way tomake it invent a value.52-week high/low and YTD are computed from the price series we already hold rather thanread from `Ticker.info`, which is an undocumented scraped endpoint that returns `{}` underload. Deriving them is both more reliable and independently checkable.

In [ ]:
print(json.dumps(equity.summary, indent=2, default=str))

## 4 · Task 1B — LLM sentiment and signal reasoningThe client is provider-agnostic with automatic failover. It queries each provider's`/models` endpoint at construction time and picks the highest-ranked model actuallyserved, rather than hardcoding an ID — `llama3-70b-8192`, the model the brief's tool tableimplies, is already deprecated on Groq. Hardcoding it means the reviewer's run fails.

In [ ]:
from common.llm_client import LLMClient, Tierllm = LLMClient(tier=Tier.REASONING, temperature=0.2)print("Run manifest:")print(json.dumps(llm.describe(), indent=2))

### 4.1 Per-headline sentimentHeadlines are classified in batches of 8 (large enough to be token-efficient, small enoughthat the model rarely drops items), then **reconciled**: any headline missing from theresponse is retried individually, and anything still missing is recorded asneutral/0.0-confidence with an explicit failure reason rather than dropped. An aggregatecomputed over a silently truncated sample is a wrong number that looks right.

In [ ]:
from llm_analysis import analyseresult = analyse(llm, equity, failure_log_path=OUTPUT_DIR / "validation_failures.jsonl")print(f"\n{'Sentiment':<10} {'Conf':<6} Headline / reason")print("─" * 118)for s in result.headline_sentiments:    print(f"{s.sentiment.value:<10} {s.confidence:<6.2f} {s.headline[:92]}")    print(f"{'':<17} └─ {s.brief_reason[:92]}")

### 4.2 Aggregate sentimentWeighted by confidence rather than a plain majority, so a hedged 0.55 classification cannotoutvote a decisive 0.95 one. Unweighted counts are kept alongside so the analyst sees both.

In [ ]:
agg = result.aggregateprint(json.dumps(agg.model_dump(mode="json"), indent=2) if agg else "No aggregate produced.")if agg and agg.degenerate_warning:    print("\n⚠️  Degenerate batch: one label AND one confidence across all headlines.")    print("    Usually means the model rubber-stamped the batch instead of reading it.")

### 4.3 Trading signalThe 15-mark criterion is *"LLM reasons over indicator combinations, not just echoesvalues."* That is normally a human judgement. Here it is **enforced in code**: the`TradingSignal` validator rejects a justification unless it (a) runs 3–5 sentences,(b) names at least two distinct indicators, and (c) contains relational language(*confirms, diverges, despite, outweighs*). A model that lists "RSI is 62, MACD is 1.4"contains no such word and is rejected, handed its own validator error, and asked again.If it still cannot synthesise after the repair attempts, a deterministic rules-basedsignal takes over and the output is **flagged as a fallback**. A degraded but honestresult beats a confident fabrication.

In [ ]:
sig = result.signalprint(f"RECOMMENDATION : {sig.recommendation.value}")print(f"CONFIDENCE     : {sig.confidence:.0%}")print(f"SOURCE         : {'⚠️  RULES-BASED FALLBACK' if result.signal_is_fallback else 'LLM (validated)'}")print(f"\nJUSTIFICATION\n{'─' * 78}\n{sig.justification}")print(f"\nKEY DRIVERS")for d in sig.key_drivers: print("  •", d)print("\nRISKS TO THIS VIEW")for r in sig.risks: print("  •", r)print(f"\nVALIDATOR REPORT\n{json.dumps(sig.quality_report(), indent=2)}")

### 4.4 Validation failuresThe brief requires validation failures to be *"caught, logged, and handled gracefully."*This is the evidence they were. An empty log is a clean run; a populated one shows thevalidation layer doing real work — which is the stronger demonstration.

In [ ]:
print(f"Failures by stage: {result.failure_summary or 'none — clean run'}\n")for f in result.validation_failures:    print(f"[{f.stage}] attempt {f.attempt} — recovered={f.recovered}")    print(f"   {f.error[:200]}\n")log_path = OUTPUT_DIR / "validation_failures.jsonl"if log_path.exists():    print(f"Persisted to {log_path.relative_to(ROOT)} "          f"({len(log_path.read_text().strip().splitlines())} entries)")

## 5 · Bonus — one-page research briefSelf-contained HTML: the chart is an embedded base64 data URI and all CSS is inline, sothe file renders identically from the repository, an email attachment, or a Drive linkwith no external requests. A brief that only renders next to its own `/images` folder isnot a deliverable.The risk disclaimer is rendered by the template rather than passed in, so it cannot beaccidentally omitted.

In [ ]:
from report import render_html_brief, render_markdown_brieffrom IPython.display import HTML, displayhtml_path = render_html_brief(equity, result, OUTPUT_DIR / f"{TICKER}_brief.html")md_path   = render_markdown_brief(equity, result, OUTPUT_DIR / f"{TICKER}_brief.md")print(f"HTML     : {html_path.relative_to(ROOT)}  ({html_path.stat().st_size / 1024:.0f} KB)")print(f"Markdown : {md_path.relative_to(ROOT)}")display(HTML(html_path.read_text(encoding="utf-8")))

## 6 · Run manifestPrinted so the reviewer can reproduce exactly this run: which provider and model servedeach request, which prompt versions were used, and how many calls it took.

In [ ]:
manifest = {    "ticker": TICKER,    "lookback_years": LOOKBACK_YEARS,    "as_of": equity.summary.get("as_of"),    "sessions_analysed": equity.summary.get("sessions_analysed"),    "headlines_classified": len(result.headline_sentiments),    "recommendation": result.signal.recommendation.value,    "signal_is_fallback": result.signal_is_fallback,    "validation_failures": result.failure_summary,    "models": result.model_manifest,    "prompts": [{"name": p["name"], "version": p["version"]} for p in result.prompt_versions],    "llm_calls": len(llm.calls),    "llm_call_detail": [        {"provider": c.provider, "model": c.model, "ok": c.ok,         "duration_s": c.duration_s, "attempt": c.attempt}        for c in llm.calls    ],}(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))print(json.dumps({k: v for k, v in manifest.items() if k != "llm_call_detail"},                 indent=2, default=str))total = sum(c.duration_s for c in llm.calls)ok = sum(c.ok for c in llm.calls)print(f"\n{len(llm.calls)} LLM calls, {ok} succeeded, {total:.1f}s total inference time.")

---## Prompt engineering notesPrompts live in `task1_financial/src/prompts.py` and contain **no business logic** — themodule imports nothing from the rest of the package. They are `string.Template` objects,not f-strings, because an f-string is evaluated at its definition site, which forces theprompt to live next to the code holding the variables — exactly the coupling the criterionis asking us to avoid. A Template is inert data that can be versioned, diffed and swappedat runtime.Every prompt carries a version and a changelog. The signal prompt is at v4:| Version | Change | Why ||---|---|---|| v1 → v2 | Added a worked pass/fail example of synthesis vs restatement | Single largest quality improvement of any change made || v2 → v3 | Added an explicit indicator weighting hierarchy | Model was letting RSI override a clear trend regime || v3 → v4 | Stated the sentence-count and relational-language rules in the prompt | The prompt and the Pydantic validator must enforce the *same* contract, or the repair loop cannot converge |System and user roles are separated deliberately: the system message carries persona,task-invariant rules and the output contract; the user message carries only this call'sdata. Mixing them makes the model treat instructions as data — and headlines areattacker-controllable in principle, so the system prompt states explicitly that headlinetext is data to classify, never instructions to follow.